#### Helper Functions

In [1]:
import os
import shutil
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from PIL import Image
from datetime import datetime
from tqdm import tqdm


# 1. Pretty Print Function
def printer(iterable: list | dict):
    """Displays Lists and Dictionaries nicely"""
    if isinstance(iterable, list):
        for item in iterable:
            print(f" - {item}")
        print()
    elif isinstance(iterable, dict):
        for key in list(iterable.keys()):
            print(f" {key} :", iterable[key])
        print()
    else:
        print(iterable)


# 2. Class Name Mapping (with cleaned names)
def getClassNames(path=r"C:\Users\Asus\Downloads\T6-AI\processedData\train"):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")
    
    folderNames = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
    
    cleanedNames = []
    for name in folderNames:
        parts = name.split()
        # Remove leading number and trailing count if present
        if parts[0][0].isdigit():
            parts.pop(0)
        if parts[-1].replace('.', '').isdigit():
            parts.pop()
        cleaned = " ".join(parts)
        cleanedNames.append(cleaned)

    classMapping = {i: cleanedNames[i] for i in range(len(cleanedNames))}
    printer(classMapping)
    return classMapping



# 4. Pad Images to Square (as a Transform Class)
class PadToSquare:
    def __call__(self, image):
        width, height = image.size
        max_side = max(width, height)
        left = (max_side - width) // 2
        right = max_side - width - left
        top = (max_side - height) // 2
        bottom = max_side - height - top
        return transforms.functional.pad(image, (left, top, right, bottom), fill=0)


#### Data-splitting code

In [3]:

# 5. Create Directories for Train/Valid/Test
def create_dirs(base_dir, class_names):
    for subset in ['train', 'valid', 'test']:
        for class_name in class_names:
            os.makedirs(os.path.join(base_dir, subset, class_name), exist_ok=True)


# 6. Split Raw Images into Train/Valid/Test
def split_and_process_images(raw_data_dir, processed_data_dir, randomState=42):
    class_names = [folder for folder in os.listdir(raw_data_dir) if os.path.isdir(os.path.join(raw_data_dir, folder))]
    create_dirs(processed_data_dir, class_names)

    for class_name in class_names:
        class_path = os.path.join(raw_data_dir, class_name)
        all_images = os.listdir(class_path)

        train_files, testval_files = train_test_split(all_images, test_size=0.2, random_state=randomState)
        val_files, test_files = train_test_split(testval_files, test_size=0.5, random_state=randomState)

        for file_list, subset in [(train_files, 'train'), (val_files, 'valid'), (test_files, 'test')]:
            for filename in file_list:
                src_path = os.path.join(class_path, filename)
                dst_path = os.path.join(processed_data_dir, subset, class_name, filename)
                if os.path.exists(src_path):  # avoid FileNotFoundError
                    Image.open(src_path).save(dst_path)

    print("✅ Images have been split and saved to:", processed_data_dir)

#### Freeze/Unfreeze according to phases

In [5]:
# 7. Freeze / Unfreeze Utility Functions
def showGrads(model):
    for name, param in model.named_parameters():
        print(f"Layer: {name}".ljust(50), f"requires_grad: {param.requires_grad}")

def freeze_model(model):
    for param in model.parameters():
        param.requires_grad = False

def unfreeze_model(model):
    for param in model.parameters():
        param.requires_grad = True

# 8. Unfreeze Last N DenseNet Blocks
def unfreeze_last_n_blocks(model, n):
    """
    Unfreezes the classifier and the last `n` dense blocks of DenseNet.
    """
    freeze_model(model)

    # Always unfreeze classifier
    for name, param in model.classifier.named_parameters():
        param.requires_grad = True

    # Unfreeze last n dense blocks
    block_names = ['denseblock1', 'denseblock2', 'denseblock3', 'denseblock4']
    selected_blocks = block_names[-n:] if n > 0 else []

    for name, param in model.features.named_parameters():
        if any(block in name for block in selected_blocks):
            param.requires_grad = True

    print(f"✅ Unfrozen: {selected_blocks + ['classifier']}")


#### Image Transforms & Data Loaders

In [7]:
def getClassNames(path=r"C:\Users\Asus\Downloads\T6-AI\processedData\train"):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")
    
    folderNames = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
    classMapping = {i: folderNames[i] for i in range(len(folderNames))}
    printer(classMapping)
    return classMapping


In [12]:

import random
from collections import Counter
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader

train_path = r"C:\Users\Asus\Downloads\T6-AI\processedData\train"
valid_path = r"C:\Users\Asus\Downloads\T6-AI\processedData\valid"
test_path  = r"C:\Users\Asus\Downloads\T6-AI\processedData\test"

# 1) Pad‐to‐square transform (reuse your existing PadToSquare if already defined)
class PadToSquare:
    def __call__(self, image):
        w, h = image.size
        max_dim = max(w, h)
        pad_w = (max_dim - w) // 2
        pad_h = (max_dim - h) // 2
        return transforms.functional.pad(
            image,
            (pad_w, pad_h, max_dim - w - pad_w, max_dim - h - pad_h),
            fill=0
        )

# 2) Normalization stats
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# 3) Augmentation pipelines
unified_augmentation = transforms.Compose([
    PadToSquare(),
    transforms.Resize((240, 240)),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomApply([transforms.GaussianBlur(3)], p=0.15),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

val_test_transform = transforms.Compose([
    PadToSquare(),
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# 4) Class‐balancing helper
def balance_classes(indices, dataset, target_samples=3000,
                    light_classes=None, moderate_classes=None, strong_classes=None):
    samples = [(dataset.samples[i][0], dataset.samples[i][1]) for i in indices]
    balanced = []
    for cls_idx, cls_name in enumerate(dataset.classes):
        cls_samples = [(p, l) for p, l in samples if l == cls_idx]
        # downsample overrepresented
        if light_classes and cls_name in light_classes and len(cls_samples) > target_samples:
            cls_samples = random.sample(cls_samples, target_samples)
        # upsample underrepresented
        elif (moderate_classes and cls_name in moderate_classes or
              strong_classes   and cls_name in strong_classes) and len(cls_samples) < target_samples:
            cls_samples += random.choices(cls_samples, k=target_samples - len(cls_samples))
        balanced.extend(cls_samples)
    return balanced

# 5) Subset wrapper that applies transforms
class TransformedSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.base_dataset[self.indices[idx]]
        return self.transform(img), label

# ——————————————————————————————————————————————————————
# 6) Build your datasets & loaders

# a) Load the full train folder
full_train = ImageFolder(train_path)

# b) Balance only the training set
light_classes = [
    '2. Melanoma 15.75k',
    '4. Basal Cell Carcinoma (BCC) 3323',
    '5. Melanocytic Nevi (NV) - 7970'
]
moderate_classes = [
    '10. Warts Molluscum and other Viral Infections - 2103',
    '6. Benign Keratosis-like Lesions (BKL) 2624',
    '7. Psoriasis pictures Lichen Planus and related diseases - 2k'
]
strong_classes = [
    '1. Eczema 1677',
    '3. Atopic Dermatitis - 1.25k',
    '8. Seborrheic Keratoses and other Benign Tumors - 1.8k',
    '9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k'
]

all_indices = list(range(len(full_train)))
balanced_samples = balance_classes(
    all_indices, full_train,
    target_samples=3000,
    light_classes=light_classes,
    moderate_classes=moderate_classes,
    strong_classes=strong_classes
)

# c) Map file‐paths back to indices
path_to_idx = {full_train.samples[i][0]: i for i in all_indices}
train_indices = [path_to_idx[p] for p, _ in balanced_samples]

# d) Create dataset objects
train_dataset = TransformedSubset(
    full_train, train_indices, transform=unified_augmentation
)
valid_dataset = ImageFolder(valid_path, transform=val_test_transform)
test_dataset  = ImageFolder(test_path,  transform=val_test_transform)

# e) DataLoaders
batchSize = 32
train_loader = DataLoader(train_dataset, batch_size=batchSize, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batchSize, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batchSize, shuffle=False)

# f) Check your balance
print("Train class counts:", Counter([lbl for _, lbl in balanced_samples]))


Train class counts: Counter({0: 3000, 1: 3000, 3: 3000, 5: 3000, 6: 3000, 7: 3000, 8: 3000, 9: 3000, 4: 2658, 2: 2512})


#### DenseNet Setup & Training Config

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained DenseNet and update classifier
from torchvision.models import densenet121, DenseNet121_Weights

classNames = getClassNames(train_path)
num_classes = len(classNames)

weights = DenseNet121_Weights.DEFAULT
densenet = densenet121(weights=weights)

densenet.classifier = nn.Sequential(
    nn.Linear(1024, 512),         # Hidden layer 1
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(512, 256),          # Hidden layer 2
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(256, 128),          # Hidden layer 3
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(0.3),

    nn.Linear(128, len(classNames))  # Final output layer
)


# Move model to GPU/CPU
densenet = densenet.to(device)

# Compute Class Weights- Add class weights to loss to handle imbalance
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Use the balanced_samples list you created earlier
# balanced_samples is a list of (path, label) tuples
y_train = [label for _, label in balanced_samples]

# Compute class weights based on that
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

# Now define your loss with these weights
criterion = nn.CrossEntropyLoss(weight=weights_tensor)


# Optimizer and Scheduler
optimizer = optim.AdamW(densenet.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Logging setup
log_path = "modelLogs"
modelVariation = "DenseNet_AugScheduler"
timestamp = datetime.today().strftime('%Y-%m-%d %H-%M')
save_dir = f"{log_path}/{modelVariation}/{timestamp}"
os.makedirs(save_dir, exist_ok=True)
best_val_accuracy = 0.0


 0 : 1. Eczema 1677
 1 : 10. Warts Molluscum and other Viral Infections - 2103
 2 : 2. Melanoma 15.75k
 3 : 3. Atopic Dermatitis - 1.25k
 4 : 4. Basal Cell Carcinoma (BCC) 3323
 5 : 5. Melanocytic Nevi (NV) - 7970
 6 : 6. Benign Keratosis-like Lesions (BKL) 2624
 7 : 7. Psoriasis pictures Lichen Planus and related diseases - 2k
 8 : 8. Seborrheic Keratoses and other Benign Tumors - 1.8k
 9 : 9. Tinea Ringworm Candidiasis and other Fungal Infections - 1.7k



#### Training Loop (In Phases)

In [ ]:
from tqdm import tqdm
from datetime import datetime
import os
import torch

# Setup save directory and best accuracy tracker
log_path = "modelLogs"
modelVariation = "DenseNet_AugScheduler"
timestamp = datetime.today().strftime('%Y-%m-%d %H-%M')
save_dir = f"{log_path}/{modelVariation}/{timestamp}"
os.makedirs(save_dir, exist_ok=True)
best_val_accuracy = 0.0


# EarlyStopping
class EarlyStopping:
    def __init__(self, patience=3, verbose=True):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_acc, model, path):
        score = val_acc

        if self.best_score is None or score > self.best_score:
            self.best_score = score
            self.counter = 0
            torch.save(model.state_dict(), path)
            if self.verbose:
                print("\n✅ Validation improved — model saved!")
        else:
            self.counter += 1
            if self.verbose:
                print(f"\n⏳ No improvement for {self.counter} epochs.")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopper = EarlyStopping(patience=3, verbose=True)

# Progressive Unfreezing Setup
phases = [0, 1, 2, 3, 4]
lr_schedule = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5]
epochs_per_phase = 5


for phase, lr in zip(phases, lr_schedule):
    print(f"\n Phase {phase+1} — Unfreezing {phase} blocks")

    # Unfreeze blocks
    unfreeze_last_n_blocks(densenet, n=phase)
    showGrads(densenet)

    # Optimizer & scheduler for current phase
    optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, densenet.parameters()),
    lr=lr,
    weight_decay=1e-4  # L2 regularization to prevent overfitting
    )
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

    for epoch in range(epochs_per_phase):
        densenet.train()
        running_loss = 0.0
        correct = 0
        total = 0

        print(f"\n Epoch {epoch+1}/{epochs_per_phase}")
        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()

            outputs = densenet(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()

        train_acc = correct / total * 100
        avg_loss = running_loss / len(train_loader)
        print(f"Epoch Summary — Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

        # ----- VALIDATION -----
        densenet.eval()
        val_correct = 0
        val_total = 0
        val_running_loss = 0.0
        
        with torch.no_grad():
            for val_inputs, val_labels in valid_loader:
                val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
                val_outputs = densenet(val_inputs)
                val_loss = criterion(val_outputs, val_labels)  # <-- compute loss
                val_running_loss += val_loss.item()
        
                _, val_preds = torch.max(val_outputs, 1)
                val_correct += (val_preds == val_labels).sum().item()
                val_total += val_labels.size(0)
        
        val_acc = val_correct / val_total * 100
        val_avg_loss = val_running_loss / len(valid_loader)
        print(f"Validation Loss: {val_avg_loss:.4f} | Validation Acc: {val_acc:.2f}%")


        # Save best model
        # Early stopping check
        early_stopper(val_acc, densenet, f"{save_dir}/best_model.pth")
        
        # Stop training if triggered
        if early_stopper.early_stop:
            print("🛑 Early stopping triggered. Stopping training.")
            break




 Phase 1 — Unfreezing 0 blocks
✅ Unfrozen: ['classifier']
Layer: features.conv0.weight                       requires_grad: False
Layer: features.norm0.weight                       requires_grad: False
Layer: features.norm0.bias                         requires_grad: False
Layer: features.denseblock1.denselayer1.norm1.weight requires_grad: False
Layer: features.denseblock1.denselayer1.norm1.bias requires_grad: False
Layer: features.denseblock1.denselayer1.conv1.weight requires_grad: False
Layer: features.denseblock1.denselayer1.norm2.weight requires_grad: False
Layer: features.denseblock1.denselayer1.norm2.bias requires_grad: False
Layer: features.denseblock1.denselayer1.conv2.weight requires_grad: False
Layer: features.denseblock1.denselayer2.norm1.weight requires_grad: False
Layer: features.denseblock1.denselayer2.norm1.bias requires_grad: False
Layer: features.denseblock1.denselayer2.conv1.weight requires_grad: False
Layer: features.denseblock1.denselayer2.norm2.weight requires_gra

Epoch Summary — Loss: 1.1496 | Train Acc: 55.94%
Validation Loss: 0.7637 | Validation Acc: 71.09%

✅ Validation improved — model saved!

 Epoch 2/5


Epoch Summary — Loss: 0.9641 | Train Acc: 63.13%
Validation Loss: 0.7326 | Validation Acc: 72.41%

✅ Validation improved — model saved!

 Epoch 3/5


Epoch Summary — Loss: 0.8969 | Train Acc: 65.83%
Validation Loss: 0.6741 | Validation Acc: 74.59%

✅ Validation improved — model saved!

 Epoch 4/5


Epoch Summary — Loss: 0.8089 | Train Acc: 69.18%
Validation Loss: 0.6590 | Validation Acc: 75.32%

✅ Validation improved — model saved!

 Epoch 5/5


...

Epoch Summary — Loss: 0.1372 | Train Acc: 95.61%
Validation Loss: 0.5356 | Validation Acc: 84.01%

✅ Validation improved — model saved!

 Epoch 5/5


Epoch Summary — Loss: 0.1168 | Train Acc: 96.26%
Validation Loss: 0.5301 | Validation Acc: 84.64%

✅ Validation improved — model saved!

 Phase 4 — Unfreezing 3 blocks
✅ Unfrozen: ['denseblock2', 'denseblock3', 'denseblock4', 'classifier']
Layer: features.conv0.weight                       requires_grad: False
Layer: features.norm0.weight                       requires_grad: False
Layer: features.norm0.bias                         requires_grad: False
Layer: features.denseblock1.denselayer1.norm1.weight requires_grad: False
Layer: features.denseblock1.denselayer1.norm1.bias requires_grad: False
Layer: features.denseblock1.denselayer1.conv1.weight requires_grad: False
Layer: features.denseblock1.denselayer1.norm2.weight requires_grad: False
Layer: features.denseblock1.denselayer1.norm2.bias requires_grad: False
Layer: features.denseblock1.denselayer1.conv2.weight requires_grad: False
Layer: features.denseblock1.denselayer2.norm1.weight requires_grad: False
Layer: features.denseblock1.de

Epoch Summary — Loss: 0.1209 | Train Acc: 96.17%
Validation Loss: 0.5645 | Validation Acc: 84.35%

⏳ No improvement for 1 epochs.

 Epoch 2/5


Epoch Summary — Loss: 0.1043 | Train Acc: 96.69%
Validation Loss: 0.5575 | Validation Acc: 84.49%

⏳ No improvement for 2 epochs.

 Epoch 3/5


Training:  85%|███████████████████████████████████████████████████████████▎          | 772/912 [08:24<01:34,  1.49it/s]

In [ ]:
# ——— Actual losses from Phases 1–3 (so far) ———

train_losses = [
    # Phase 1
    1.1496, 0.9641, 0.8969, 0.8089, 0.7675,
    # Phase 2
    0.7889, 0.6316, 0.5377, 0.3898, 0.3236,
    # Phase 3 (to date)
    0.2895, 0.2335, 0.1943
]

val_losses = [
    # Phase 1
    0.7637, 0.7326, 0.6741, 0.6590, 0.6695,
    # Phase 2
    0.6296, 0.5799, 0.5762, 0.5798, 0.5957,
    # Phase 3 (to date)
    0.5472, 0.5306, 0.5370
]

# sanity check
assert len(train_losses) == len(val_losses), \
    f"{len(train_losses)} train vs {len(val_losses)} val points"

# plotting
import matplotlib.pyplot as plt

epochs = list(range(1, len(train_losses) + 1))

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, val_losses,   marker='s', label='Validation Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss (Phases 1–3)')
plt.xticks(epochs)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


#### Final Evaluation on Test Set

In [22]:
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch

# Ensure the path to the best model exists
model_path = Path(save_dir) / "best_model.pth"

if not model_path.exists():
    raise FileNotFoundError(f"🚫 Model not found at: {model_path}")

# Safe loading (recommended by PyTorch)
densenet.load_state_dict(torch.load(model_path, weights_only=True))
densenet.eval()

# Evaluate on test set
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in test_dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = densenet(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Print overall metrics
print("Test Metrics Summary:")
print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred, average='weighted'):.4f}")
print(f"F1 Score : {f1_score(y_true, y_pred, average='weighted'):.4f}")

# Print classification results
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=list(classNames.values())))


Test Metrics Summary:
Accuracy : 0.8124
Precision: 0.8209
Recall   : 0.8124
F1 Score : 0.8062
Classification Report:
                                                                  precision    recall  f1-score   support

                                                  1. Eczema 1677       0.74      0.71      0.73       168
           10. Warts Molluscum and other Viral Infections - 2103       0.81      0.72      0.76       211
                                              2. Melanoma 15.75k       0.96      1.00      0.98       314
                                    3. Atopic Dermatitis - 1.25k       0.54      0.66      0.59       126
                              4. Basal Cell Carcinoma (BCC) 3323       0.93      0.75      0.83       333
                                 5. Melanocytic Nevi (NV) - 7970       0.81      1.00      0.89       797
                     6. Benign Keratosis-like Lesions (BKL) 2624       0.89      0.46      0.61       208
   7. Psoriasis pictures Lichen Pl

#### Predict Unseen Images + Confidence Score

In [32]:
#Data Preprocessing
from torchvision import transforms
from PIL import Image
import torch
import torch.nn.functional as F

# ImageNet normalization
imageNet_mean = [0.485, 0.456, 0.406]
imageNet_std = [0.229, 0.224, 0.225]
imgSize = (224, 224)

# Transform
transform_single = transforms.Compose([
    transforms.Lambda(PadToSquare()),
    transforms.Resize(imgSize),
    transforms.ToTensor(),
    transforms.Normalize(mean=imageNet_mean, std=imageNet_std),
])


In [34]:
#Predict images from custom_test file
import os
import torch.nn.functional as F


test_image_dir = r"C:\Users\Asus\Downloads\T6-AI\testimgs"
image_files = os.listdir(test_image_dir)

densenet.eval()  # Make sure model is in eval mode

transform_single = transforms.Compose([
    PadToSquare(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


for filename in image_files:
    image_path = os.path.join(test_image_dir, filename)
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform_single(image).unsqueeze(0).to(device)  # Add batch dim

    with torch.no_grad():
        output = densenet(input_tensor)
        probs = F.softmax(output, dim=1)
        confidence, pred_idx = torch.max(probs, 1)
        predicted_class = list(classNames.values())[pred_idx.item()]
        confidence_score = confidence.item() * 100

        print(f"{filename:30} → Predicted: {predicted_class:50} |Confidence: {confidence_score:.2f}%")


atopic (1).jpg                 → Predicted: 3. Atopic Dermatitis - 1.25k                       |Confidence: 98.00%
basal test (1).jpg             → Predicted: 5. Melanocytic Nevi (NV) - 7970                    |Confidence: 99.96%
benign test (1).jpg            → Predicted: 5. Melanocytic Nevi (NV) - 7970                    |Confidence: 99.98%
ezema_test (1).jpg             → Predicted: 1. Eczema 1677                                     |Confidence: 98.16%
melanoma_test (1).jpg          → Predicted: 2. Melanoma 15.75k                                 |Confidence: 99.95%
melanyontic test (1).jpg       → Predicted: 5. Melanocytic Nevi (NV) - 7970                    |Confidence: 99.92%
psiosrasis test (1).jpg        → Predicted: 7. Psoriasis pictures Lichen Planus and related diseases - 2k |Confidence: 82.11%
serra test (1).jpg             → Predicted: 8. Seborrheic Keratoses and other Benign Tumors - 1.8k |Confidence: 88.05%
Tinea Ringworm Candidiasis and other Fungal Infections_test (1).j